# MP2: OSS activity analysis — bdowlin2

Part 1: WoC retrieval and source validation. See `bdowlin2_vis.ipynb` for monthly activity and gap analysis.

## Sources and method

Commit membership and historical commit details come **strictly from World of Code**. The Python retrieval helper calls WoC's HTTP API directly, avoiding the Windows build dependency of the optional `python-woc` package. It uses the same documented `p2c` map and `commit` object endpoints. No GitHub commit data is substituted into the WoC summary.

- [WoC remote API documentation](https://worldofcode.org/docs/#/guide_remote)
- [Official Python remote client](https://github.com/ssc-oscar/python-woc/blob/main/woc/remote.py)
- WoC lists the commits associated with a project across its indexed history. This may differ from GitHub's current default branch. Exact author strings are counted; aliases are not merged.
- Author timestamps, in UTC, determine monthly WoC activity. Empty messages are allowed. SHA/project pairs must be unique, and every expected SHA must have details before analysis runs.
- The README describes an older January 2025 snapshot, but some retrieved records are later. We report actual returned date ranges and do not infer present abandonment from a project history endpoint.
- GitHub is used separately for stars, forks, recent themes, status, and supporting evidence. Stars/forks are current retrieval-time values, not reconstructed September 30 counts.
- **Status cutoff: September 30, 2026, 23:59:59 UTC.** Active means the latest default-branch commit returned by GitHub on/before the cutoff is on/after April 1, 2026. This is the user's approved replacement for the README's 2025 cutoff. GitHub uses committer dates for these API filters.
- The ten most recent GitHub commits on/before the cutoff include the latest commit; fewer are retained if fewer are returned. Default-branch history need not equal WoC's project history.


In [1]:
from pathlib import Path
import json, sqlite3
import pandas as pd
from IPython.display import display, Markdown
from mp2_retrieve import ROOT, CACHE, NETID, FIELDS, projects, retrieve
from mp2_github import collect

assert (ROOT / 'net2prj.csv').exists(), 'Run from the MiniProject2 folder.'
assigned = pd.DataFrame(projects())
display(assigned)


,netID,WoC,GH
0,bdowlin2,avehtari_bda_course_aalto,https://github.com/avehtari/BDA_course_Aalto
1,bdowlin2,eberharf_cfl,https://github.com/eberharf/cfl
2,bdowlin2,openda-association_openda,https://github.com/OpenDA-Association/OpenDA
3,bdowlin2,openms_openms,https://github.com/OpenMS/OpenMS
4,bdowlin2,vatlab_sos,https://github.com/vatlab/SoS
5,bdowlin2,jayggg_ngstents,https://github.com/jayggg/ngstents
6,bdowlin2,inqwire_sqir,https://github.com/inQWIRE/SQIR
7,bdowlin2,tee-lab_pyddsde,https://github.com/tee-lab/pyddsde
8,bdowlin2,jgcri_gcam-core,https://github.com/JGCRI/gcam-core
9,bdowlin2,windweller_disextract,https://github.com/windweller/DisExtract


## Retrieve and resume

Run `python mp2_retrieve.py`, or set `RUN_NETWORK = True` below. Requests contain ten commits and are separated by two-second waits. Failed requests use bounded retries and respect Retry-After. Two consecutive failed batches stop the run. Every successful batch is committed to SQLite. Rerunning skips saved details. The final five-column semicolon CSV is replaced only after all projects pass completeness checks.

The original notebook and hash-only CSV are preserved in `work/original/`. The original CSV at the project root remains a legacy hash list until full retrieval succeeds; do not treat it as a complete summary.

In [2]:
RUN_NETWORK = False  # Set True to resume WoC retrieval and refresh GitHub metadata.
if RUN_NETWORK:
    retrieve()
    collect()
else:
    print('Using saved checkpoints. No network retrieval requested by this run.')

Using saved checkpoints. No network retrieval requested by this run.


In [3]:
def coverage_table():
    database = CACHE / 'commits.sqlite'
    cached = set()
    if database.exists():
        with sqlite3.connect(database) as connection:
            cached = {row[0] for row in connection.execute('SELECT sha FROM commits')}
    rows = []
    for project in projects():
        source = CACHE / (project['WoC'] + '_shas.json')
        expected = set(json.loads(source.read_text())) if source.exists() else set()
        rows.append({'Project': project['WoC'], 'Expected commits': len(expected) if source.exists() else None,
                     'Details saved': len(expected & cached), 'Missing details': len(expected - cached) if source.exists() else None,
                     'Complete': source.exists() and bool(expected) and expected <= cached})
    return pd.DataFrame(rows)

coverage = coverage_table()
display(coverage)
print('All projects complete:', bool(coverage['Complete'].all()))


,Project,Expected commits,Details saved,Missing details,Complete
0,avehtari_bda_course_aalto,2272,2272,0,True
1,eberharf_cfl,1138,1138,0,True
2,openda-association_openda,3209,3202,7,False
3,openms_openms,54864,299,54565,False
4,vatlab_sos,8834,8028,806,False
5,jayggg_ngstents,454,454,0,True
6,inqwire_sqir,1338,1338,0,True
7,tee-lab_pyddsde,233,233,0,True
8,jgcri_gcam-core,10688,60,10628,False
9,windweller_disextract,559,559,0,True


All projects complete: False


## Required project statistics

The text cells below report the retrieved GitHub metadata and WoC commit counts. Author counts and date ranges are reported only for fully retrieved projects. Partial samples must not be interpreted as full-project statistics.

In [4]:
from mp2_analysis import load_complete
if coverage['Complete'].any():
    summary = load_complete(completed_only=True)
    woc_stats = summary.groupby('project_wocid').agg(ncommits=('commit_sha1', 'nunique'), nauthors=('author', 'nunique'), min_time=('date', 'min'), max_time=('date', 'max'))
    display(woc_stats)
    print('Only projects with exact, complete WoC SHA coverage are included above.')
else:
    print('WoC author counts and date ranges are unavailable until a project is fully retrieved.')

,ncommits,nauthors,min_time,max_time
project_wocid,,,,
avehtari_bda_course_aalto,2272,67,2018-09-07 14:11:11+00:00,2025-11-03 15:40:44+00:00
eberharf_cfl,1138,14,2020-07-17 21:53:13+00:00,2025-05-07 22:32:37+00:00
inqwire_sqir,1338,34,2018-12-20 22:11:21+00:00,2025-07-30 02:27:23+00:00
jayggg_ngstents,454,11,2020-06-29 20:41:15+00:00,2025-06-10 12:50:41+00:00
tee-lab_pyddsde,233,7,2020-03-30 14:40:44+00:00,2021-10-29 06:08:17+00:00
windweller_disextract,559,9,2017-10-29 23:56:48+00:00,2025-07-13 15:34:42+00:00


Only projects with exact, complete WoC SHA coverage are included above.


### avehtari_bda_course_aalto

[GitHub repository](https://github.com/avehtari/BDA_course_Aalto)

- GitHub stars: **2282**; forks: **566** (retrieved 2026-10-08).
- Latest GitHub commit on/before the cutoff: **2026-09-29T11:32:50Z**.
- WoC commit count: **2,272** distinct hashes.
- WoC author count: **67 distinct author strings**; minimum time: **2018-09-07T14:11:11+00:00**; maximum time: **2025-11-03T15:40:44+00:00**. Complete SHA coverage verified.

### eberharf_cfl

[GitHub repository](https://github.com/eberharf/cfl)

- GitHub stars: **19**; forks: **1** (retrieved 2026-10-08).
- Latest GitHub commit on/before the cutoff: **2026-05-05T21:52:43Z**.
- WoC commit count: **1,138** distinct hashes.
- WoC author count: **14 distinct author strings**; minimum time: **2020-07-17T21:53:13+00:00**; maximum time: **2025-05-07T22:32:37+00:00**. Complete SHA coverage verified.

### openda-association_openda

[GitHub repository](https://github.com/OpenDA-Association/OpenDA)

- GitHub stars: **104**; forks: **34** (retrieved 2026-10-08).
- Latest GitHub commit on/before the cutoff: **2026-07-10T13:57:20Z**.
- WoC commit count: **3,209** distinct hashes.
- WoC author count, minimum time, maximum time: **unavailable until commit-detail retrieval completes**.

### openms_openms

[GitHub repository](https://github.com/OpenMS/OpenMS)

- GitHub stars: **623**; forks: **445** (retrieved 2026-10-08).
- Latest GitHub commit on/before the cutoff: **2026-09-30T17:00:08Z**.
- WoC commit count: **54,864** distinct hashes.
- WoC author count, minimum time, maximum time: **unavailable until commit-detail retrieval completes**.

### vatlab_sos

[GitHub repository](https://github.com/vatlab/SoS)

- GitHub stars: **286**; forks: **44** (retrieved 2026-10-08).
- Latest GitHub commit on/before the cutoff: **2026-09-09T18:36:17Z**.
- WoC commit count: **8,834** distinct hashes.
- WoC author count: **55 distinct author strings**; minimum time: **2016-02-13T00:46:46+00:00**; maximum time: **2025-11-06T14:29:35+00:00**. Complete SHA coverage verified.

### jayggg_ngstents

[GitHub repository](https://github.com/jayggg/ngstents)

- GitHub stars: **9**; forks: **6** (retrieved 2026-10-08).
- Latest GitHub commit on/before the cutoff: **2025-11-27T00:17:25Z**.
- WoC commit count: **454** distinct hashes.
- WoC author count: **11 distinct author strings**; minimum time: **2020-06-29T20:41:15+00:00**; maximum time: **2025-06-10T12:50:41+00:00**. Complete SHA coverage verified.

### inqwire_sqir

[GitHub repository](https://github.com/inQWIRE/SQIR)

- GitHub stars: **105**; forks: **26** (retrieved 2026-10-08).
- Latest GitHub commit on/before the cutoff: **2025-07-30T02:27:23Z**.
- WoC commit count: **1,338** distinct hashes.
- WoC author count: **34 distinct author strings**; minimum time: **2018-12-20T22:11:21+00:00**; maximum time: **2025-07-30T02:27:23+00:00**. Complete SHA coverage verified.

### tee-lab_pyddsde

[GitHub repository](https://github.com/tee-lab/pyddsde)

- GitHub stars: **98**; forks: **12** (retrieved 2026-10-08).
- Latest GitHub commit on/before the cutoff: **2026-06-10T04:11:37Z**.
- WoC commit count: **233** distinct hashes.
- WoC author count: **7 distinct author strings**; minimum time: **2020-03-30T14:40:44+00:00**; maximum time: **2021-10-29T06:08:17+00:00**. Complete SHA coverage verified.

### jgcri_gcam-core

[GitHub repository](https://github.com/JGCRI/gcam-core)

- GitHub stars: **432**; forks: **226** (retrieved 2026-10-08).
- Latest GitHub commit on/before the cutoff: **2026-06-01T21:04:27Z**.
- WoC commit count: **10,688** distinct hashes.
- WoC author count, minimum time, maximum time: **unavailable until commit-detail retrieval completes**.

### windweller_disextract

[GitHub repository](https://github.com/windweller/DisExtract)

- GitHub stars: **0**; forks: **0** (retrieved 2026-10-08).
- Latest GitHub commit on/before the cutoff: **2025-07-13T15:34:42Z**.
- WoC commit count: **559** distinct hashes.
- WoC author count: **9 distinct author strings**; minimum time: **2017-10-29T23:56:48+00:00**; maximum time: **2025-07-13T15:34:42+00:00**. Complete SHA coverage verified.

## Validation and limitations

The SHA manifest covers all ten assigned projects. Complete-project checks compare exact SHA sets, duplicate keys, author fields, SHA formats, and timestamp parsing. `bdowlin2_source_comparison.csv` reports WoC counts alongside GitHub default-branch counts as of the cutoff. Differences are retained rather than forcing equality across different history scopes. In particular, DisExtract exposes only three current default-branch commits compared with 559 WoC-associated commits.

The README describes an older WoC snapshot, but the retrieved data includes some 2025 commits beyond January. Use each project's actual returned range; no uniform live snapshot date has been verified. Missing commit details are never treated as zero activity.